<a href="https://colab.research.google.com/github/deltorobarba/astrophysics/blob/main/excited_state_calculations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Excited-State Calculations**

Python example for excited-state calculations using quantum chemistry tools. One common approach for excited-state calculations is to use Variational Quantum Eigensolver (VQE) methods or time-dependent simulations. Here, I will use the PennyLane library, which can interface with OpenFermion and other quantum chemistry packages for excited-state calculations. This example shows how to calculate excited states using the state-specific VQE method for molecular systems.

Example: Excited-State VQE Calculation for a Hydrogen Molecule (H₂)

In this example, we'll use PennyLane to calculate the ground and first excited state energies of a hydrogen molecule (H₂).

In [11]:
!pip install pennylane -q

In [12]:
!pip install openfermion -q

In [ ]:
!pip install pyscf -q

https://docs.pennylane.ai/en/stable/code/qml_qchem.html

In [16]:
import pennylane as qml
from pennylane import numpy as np
#from pennylane_qchem import molecule, qubit_hamiltonian, qubit_observable
from openfermion.chem import MolecularData
#from openfermionpyscf import run_pyscf

# Define the molecular system
geometry = [("H", (0.0, 0.0, -0.6614)), ("H", (0.0, 0.0, 0.6614))]  # Hydrogen molecule
charge = 0
spin = 0  # Multiplicity = 2S + 1, so spin = 0 corresponds to singlet state
basis = 'sto-3g'

# Generate molecular data using OpenFermion
molecule_data = MolecularData(geometry, basis, charge, spin)
molecule_data = run_pyscf(molecule_data, run_scf=True, run_fci=True)

# Obtain molecular Hamiltonian
hamiltonian = molecule_data.get_molecular_hamiltonian()
qubit_ham = qubit_hamiltonian(hamiltonian)

# Set up a quantum device
n_qubits = len(qubit_ham.wires)
dev = qml.device("default.qubit", wires=n_qubits)

# Define ansatz for the VQE
def vqe_ansatz(params, wires):
    qml.BasisState(np.array([1, 1, 0, 0]), wires=wires)
    qml.DoubleExcitation(params[0], wires=[0, 1, 2, 3])
    qml.SingleExcitation(params[1], wires=[0, 1])

# Define cost function for VQE
def cost_fn(params):
    return qml.ExpvalCost(vqe_ansatz, qubit_ham, dev, optimize=True)(params)

# Initialize parameters
params = np.random.normal(0, np.pi, 2)

# Optimize the parameters for ground state
optimizer = qml.GradientDescentOptimizer(stepsize=0.4)
max_iterations = 100
conv_tol = 1e-6

for n in range(max_iterations):
    params, energy = optimizer.step_and_cost(cost_fn, params)
    if n % 10 == 0:
        print(f"Iteration = {n}, Energy = {energy:.8f}")
    if np.abs(cost_fn(params) - energy) < conv_tol:
        break

# Final energy corresponds to the ground state
print(f"Final Ground State Energy = {energy:.8f} Ha")

# Excited-State Calculation (Simple Variational State)
def excited_ansatz(params, wires):
    qml.BasisState(np.array([0, 1, 1, 0]), wires=wires)
    qml.DoubleExcitation(params[0], wires=[0, 1, 2, 3])
    qml.SingleExcitation(params[1], wires=[0, 1])

# Define cost function for excited state
excited_cost_fn = qml.ExpvalCost(excited_ansatz, qubit_ham, dev, optimize=True)

# Initialize new parameters for excited state
params_excited = np.random.normal(0, np.pi, 2)

# Optimize for excited state
for n in range(max_iterations):
    params_excited, energy_excited = optimizer.step_and_cost(excited_cost_fn, params_excited)
    if n % 10 == 0:
        print(f"Iteration = {n}, Excited-State Energy = {energy_excited:.8f}")
    if np.abs(excited_cost_fn(params_excited) - energy_excited) < conv_tol:
        break

# Final energy corresponds to the excited state
print(f"Final Excited-State Energy = {energy_excited:.8f} Ha")


MoleculeNameError: Invalid spin multiplicity provided.

1. **Molecular Definition**: The molecular geometry for H₂ is defined, and OpenFermion with PySCF is used to generate the molecular Hamiltonian.
2. **VQE Ansatz**: We use a simple ansatz that includes single and double excitations for VQE. You can customize this based on the molecule's complexity.
3. **Ground-State Calculation**: The VQE optimizes the parameters to find the ground state energy of the molecule.
4. **Excited-State Ansatz**: A similar variational ansatz is defined for the excited state, and the energy is optimized to find the first excited state energy.

**Next Steps:**
- You can extend this approach to more complex molecules by changing the molecular geometry and basis set.
- You can also use more sophisticated ansätze like Unitary Coupled Cluster (UCC) for more accurate results.
- PennyLane also supports other quantum devices such as IBMQ, which can be integrated for running on real quantum hardware.